In [80]:
import re

# get book pdf paths
ai_pdf_path = "../../files/ai/inputs/ai_index.pdf"
ai_book_path = "../../files/ai/inputs/ai.pdf"
ai_tex_path = "../../files/ai/TOC FIX/ai_pg_sep.tex"
alg_pdf_path = "../../files/algorithms/inputs/algorithms_index.pdf"
alg_book_path = "../../files/algorithms/inputs/algorithms.pdf"
alg_tex_path = "../../files/algorithms/TOC FIX/algorithms_pg_sep.tex"
assmb_pdf_path = "../../files/assembly/inputs/assembly-index.pdf"
assmb_book_path = "../../files/assembly/inputs/assembly.pdf"
assmb_tex_path = "../../files/assembly/TOC FIX/assembly_pg_sep.tex"
cyber_pdf_path = "../../files/cybersec/inputs/cybersec-index.pdf"
cyber_book_path = "../../files/cybersec/inputs/cybersec.pdf"
cyber_tex_path = "../../files/cybersec/TOC FIX/cybersec_pg_sep.tex"
ds_pdf_path = "../../files/data-science/inputs/data-science-index.pdf"
ds_book_path = "../../files/data-science/inputs/data-science.pdf"
ds_tex_path = "../../files/data-science/TOC FIX/data-science_pg_sep.tex"

import fitz  # PyMuPDF


In [12]:
import fitz  # PyMuPDF
bk = "ai"
pdf_path = ai_pdf_path
doc = fitz.open(pdf_path)
final_list = []
x0 = -1
y0 = -1
x1 = -1
y1 = -1 # working coordinates
current_line = -1
line_string = ""
current_block = -1
for page_num, page in enumerate(doc, start=1):
    print(f"--- Page {page_num} ---")
    text = page.get_text()
    # print(text)
    text = page.get_text("words")  # extracts text in reading order
    current_line = -1
    line_string = ""
    current_block = -1
    
    page_data = []
    for word in text:
        x0_, y0_, x1_, y1_, word_str, block_no, line_no, word_no = word
        print(word)
        # sometimes pymupdf gives different lin index to things that are on same line, manuakl check with x and y coords
        flag = ((abs(y0_ - y0) < 5) and x0_ -x1 <= 50) or line_no == current_line   # flag true means same line

        if block_no != current_block or (not flag ):
            # save current line before moving to next
            if current_line != -1:
                page_data.append((x0, y0, x1, y1, line_string))
            
            current_block = block_no
            current_line = line_no
            line_string = ""
            x0, y0, x1, y1 = x0_, y0_, x1_, y1_
        
        # appending word and managing coordinates
        line_string += word_str + " "
        x1 = max(x1, x1_)
        y1 = max(y1, y1_)
        x0 = min(x0, x0_)
        y0 = min(y0, y0_)

    page_data.append((x0, y0, x1, y1, line_string))
    final_list.append((page_num, page_data))
        

--- Page 1 ---
(53.85820007324219, 67.56885528564453, 93.5017318725586, 83.5091552734375, 'Index', 0, 0, 0)
(53.85820007324219, 190.1238555908203, 59.93836975097656, 198.59205627441406, 'A', 1, 0, 0)
(53.85820007324219, 198.47933959960938, 100.3957290649414, 209.71176147460938, 'A⋆-algorithm,', 1, 1, 0)
(103.29185485839844, 201.2401123046875, 118.1112060546875, 209.70831298828125, '107,', 1, 1, 1)
(120.92264556884766, 201.2401123046875, 133.62493896484375, 209.70831298828125, '273', 1, 1, 2)
(53.85449981689453, 211.16485595703125, 78.79335021972656, 219.633056640625, 'Action,', 1, 2, 0)
(81.63019561767578, 211.16485595703125, 92.24932098388672, 219.633056640625, '97,', 1, 2, 1)
(95.06076049804688, 211.16485595703125, 103.52896118164062, 219.633056640625, '98', 1, 2, 2)
(53.846031188964844, 221.089599609375, 89.17536163330078, 229.55780029296875, 'Activation', 1, 3, 0)
(92.05455017089844, 221.089599609375, 122.10819244384766, 229.55780029296875, 'function,', 1, 3, 1)
(124.93656921386719

In [ ]:
final_list

In [13]:
def is_pattern(text):
    pattern = re.compile(
    r"""
    ^(.+?),                # group 1: text (greedy, up to last comma before numbers)
    \s*                    # optional spaces
    ((?:\d+(?:–\d+)?       # one number or range (e.g., 150 or 204–208)
        (?:,\s*\d+(?:–\d+)?)*))  # optionally more numbers/ranges separated by commas
    $                      # end of string
    """,
    re.VERBOSE
    )

    pattern = re.compile(
        r"""
        ^\s*                                # optional leading spaces
        (?P<entry>.+?)                      # capture entry text (lazy)
        # \s*,?\s*                            # optional comma before numbers
        \s*,?\s+                     # **at least one space** before numbers, optional comma allowed here
        ((?P<pages>(\d+(?:\s*[–-]\s*\d+)?   # a number or range
                (?:\s*,\s*\d+(?:\s*[–-]\s*\d+)?)*   # more numbers/ranges separated by commas
                )
        )
        |
        (?:see\s+                  # OR the cross-reference "see" keyword (case-insensitive)
            (?P<see_ref>.+)         # cross-reference text (rest of line)
        ))
        
        \s*$                               # optional trailing spaces
        """,
        re.IGNORECASE | re.VERBOSE
    )
    if '©' in text:
        return False, None, None, None
    match = pattern.match(text)
    if match:
        text= match.group("entry")
        pages = match.group("pages")
        see_ref = match.group("see_ref")
        return True, text, pages, see_ref
    else:
        return False, None, None, None

In [ ]:
examples = [
    "Bayes’ theorem, 150, 205, 299",
    "Babbage, Charles, 57, 90",
    "cmpsb, 198, 204–208, 324",
    "eflags register, 9, 49, 66–67",
    "0-day attack 295, 323",
    "350",
    '© The Editor(s) (if applicable) and The Author(s), under exclusive license ',
    'something  See otherthing'
]

for line in examples:
    chk, _, _,_ = is_pattern(line)
    print(f"Line: {line} | Match: {chk}")

In [14]:
columnar_data = [] # (page number,  columns = {column_dims = [], column_lines = []} )

current_column = [-1,-1,-1,-1]
page_columns = []
column_data =[]
counter = 2
is_valid = False
is_footer = False
for page_num, page_data in final_list:
    is_valid = False
    print(f"Processing page {page_num}")
    for line in page_data:
        x0, y0, x1, y1, line_string = line

        # print(f"Line: {line_string.strip()} at ({x0}, {y0}, {x1}, {y1})")
        # # check if line fits in current column
        if current_column[0] == -1:
            # initialize column
            print("New column detected")
            print(f"Current column: {current_column}")
            current_column = [x0,y0,x1,y1]
            print(f"Starting new column with line at ({x0}, {y0}, {x1}, {y1}) with text: {line_string.strip()}")
            if '©' in line_string:
                is_footer = True
            is_valid = False
            column_data = []
            column_data.append(line)
            chk, _, _,_ = is_pattern(line_string.strip())
            is_valid = is_valid or chk

        else:
            # check if line fits in current column

            if ((current_column[0] <= x0 and x0  <= current_column[2] ) or (current_column[0] <= x1 and x1 <= current_column[2]) or (x0 <= current_column[0] and current_column[2] <= x1)) and \
                (current_column[0]- x0 <= 50):
                # print("Fits in current column")
                # update column coordinates
                current_column[0] = min(current_column[0], x0)
                current_column[2] = max(current_column[2], x1)
                current_column[1] = min(current_column[1], y0)
                current_column[3] = max(current_column[3], y1)
                column_data.append(line)
                chk, _, _,_ = is_pattern(line_string.strip())
                if '©' in line_string:
                    is_footer = True
                is_valid = is_valid or chk
            else:
                # save current column and start new one
                print("New column detected")
                print(f"Current column: {current_column}")
                print(f"Starting new column with line at ({x0}, {y0}, {x1}, {y1}) with text: {line_string.strip()}")
                if is_valid and not is_footer:
                    page_columns.append({"dimensions" : current_column, "lines": column_data})
                current_column = [x0,y0,x1,y1]
                is_valid = False
                is_footer = False
                column_data = []
                column_data.append(line)
                chk, _, _, _ = is_pattern(line_string.strip())
                is_valid = is_valid or chk
                if '©' in line_string:
                    is_footer = True
    # save last column of the page
    if is_valid and not is_footer:
        page_columns.append({"dimensions" : current_column, "lines": column_data})
    is_valid = False
    is_footer = False
    print(f"End of page {page_num}, columns: {page_columns}")
    if current_column[0] != -1:
        columnar_data.append([page_num, page_columns])
        current_column = [-1,-1,-1,-1]
    page_columns = []

    



Processing page 1
New column detected
Current column: [-1, -1, -1, -1]
Starting new column with line at (53.85820007324219, 67.56885528564453, 93.5017318725586, 83.5091552734375) with text: Index
New column detected
Current column: [53.83757400512695, 67.56885528564453, 204.9525909423828, 557.0366821289062]
Starting new column with line at (225.63970947265625, 190.15101623535156, 231.27952575683594, 198.6192169189453) with text: B
New column detected
Current column: [225.63131713867188, 190.15101623535156, 385.54473876953125, 557.0447387695312]
Starting new column with line at (53.848419189453125, 591.7719116210938, 210.9843292236328, 601.43505859375) with text: © Springer International Publishing AG 2017
New column detected
Current column: [53.848419189453125, 591.7719116210938, 294.7686767578125, 621.3353271484375]
Starting new column with line at (372.5152587890625, 590.9251098632812, 385.47161865234375, 599.393310546875) with text: 351
End of page 1, columns: [{'dimensions': [53.83

In [15]:
page_columns = {}
for page_data in columnar_data:
    print(f"Page {page_data[0]} has {len(page_data[1])} columns.")
    col_list = []

    for column in page_data[1]:
        print("Column Coordinates:", column["dimensions"])
        
        col_list.append(column["dimensions"])
    page_columns[page_data[0]] = col_list

Page 1 has 2 columns.
Column Coordinates: [53.83757400512695, 67.56885528564453, 204.9525909423828, 557.0366821289062]
Column Coordinates: [225.63131713867188, 190.15101623535156, 385.54473876953125, 557.0447387695312]
Page 2 has 2 columns.
Column Coordinates: [53.82435607910156, 59.054290771484375, 206.9547882080078, 563.7483520507812]
Column Coordinates: [225.60177612304688, 59.102935791015625, 375.10589599609375, 563.8623046875]
Page 3 has 2 columns.
Column Coordinates: [53.815879821777344, 59.054290771484375, 188.33322143554688, 563.8428955078125]
Column Coordinates: [225.6356201171875, 59.087432861328125, 369.4696044921875, 563.946044921875]
Page 4 has 2 columns.
Column Coordinates: [53.347999572753906, 58.97477340698242, 213.67852783203125, 593.5165405273438]
Column Coordinates: [225.6088409423828, 59.105377197265625, 385.5596618652344, 593.5892333984375]
Page 5 has 2 columns.
Column Coordinates: [53.85816192626953, 59.054290771484375, 213.60232543945312, 583.6337280273438]
Colum

In [16]:
diffs = []
for page_data in columnar_data:
    print(f"Page {page_data[0]} has {len(page_data[1])} columns.")
    for column in page_data[1]:
        print("Column Coordinates:", column["dimensions"])
        x_main = column["dimensions"][0]
        for line in column["lines"]:
            print(" Line:", line)
            x_line = line[0]
            diffs.append(abs(x_main - x_line))


Page 1 has 2 columns.
Column Coordinates: [53.83757400512695, 67.56885528564453, 204.9525909423828, 557.0366821289062]
 Line: (53.85820007324219, 67.56885528564453, 93.5017318725586, 83.5091552734375, 'Index ')
 Line: (53.85820007324219, 190.1238555908203, 59.93836975097656, 198.59205627441406, 'A ')
 Line: (53.85820007324219, 198.47933959960938, 133.62493896484375, 209.71176147460938, 'A⋆-algorithm, 107, 273 ')
 Line: (53.85449981689453, 211.16485595703125, 103.52896118164062, 219.633056640625, 'Action, 97, 98 ')
 Line: (53.846031188964844, 221.089599609375, 155.3289337158203, 229.55780029296875, 'Activation function, 248, 258 ')
 Line: (53.84602355957031, 231.01434326171875, 99.7860107421875, 239.4825439453125, 'Actuators, 17 ')
 Line: (53.84602355957031, 240.9390869140625, 193.76608276367188, 249.40728759765625, 'Adaptive Resonance Theory (ART), 286 ')
 Line: (53.84601593017578, 250.86383056640625, 127.3245849609375, 259.33203125, 'Admissible, 107, 109 ')
 Line: (53.84601593017578, 

In [17]:
diffs

[0.020626068115234375,
 0.020626068115234375,
 0.020626068115234375,
 0.016925811767578125,
 0.008457183837890625,
 0.008449554443359375,
 0.008449554443359375,
 0.008441925048828125,
 0.008441925048828125,
 36.006771087646484,
 11.965557098388672,
 11.965557098388672,
 11.965557098388672,
 11.965557098388672,
 11.965557098388672,
 11.965557098388672,
 11.965557098388672,
 11.97402572631836,
 11.97403335571289,
 11.97403335571289,
 11.97403335571289,
 0.008472442626953125,
 0.008472442626953125,
 0.0084686279296875,
 0.008464813232421875,
 3.814697265625e-06,
 3.814697265625e-06,
 0.0,
 3.814697265625e-06,
 3.814697265625e-06,
 0.008480072021484375,
 0.016941070556640625,
 0.016231536865234375,
 0.016239166259765625,
 0.016231536865234375,
 0.0162353515625,
 0.0162353515625,
 0.016231536865234375,
 0.008392333984375,
 0.008392333984375,
 0.008392333984375,
 11.965499877929688,
 0.008392333984375,
 0.016845703125,
 0.016845703125,
 0.016845703125,
 0.016845703125,
 36.01519775390625,
 1

In [18]:
diffs

# remove diffs less than 1
diffs = [d for d in diffs if d >= 1]
diffs

# sort diffs
diffs.sort()
diffs

# check first and last diff to see range
min_diff = diffs[0]
max_diff = diffs[-1]
min_diff, max_diff
# subtract min_diff from max_diff
range_diff = max_diff - min_diff
range_diff

# if range is greater than 2, then we have subindexes and continuations, else only subindexes
if range_diff > 2:
    print("Both subindexes and continuations detected")
    # separate diffs into two groups based on a threshold
    subindexes = [d for d in diffs if abs(d - min_diff) <= 2]
    continuations = [d for d in diffs if abs(d - max_diff) <= 2]
    # average diff for subindexes
    avg_diff_subindexes = sum(subindexes) / len(subindexes)
    avg_diff_subindexes
    # average diff for continuations
    avg_diff_continuations = sum(continuations) / len(continuations)
    avg_diff_continuations
    print(f"Avg Subindexes Diff: {avg_diff_subindexes}, Avg Continuations Diff: {avg_diff_continuations}")
else:
    print("Only subindexes detected")
    # average diff
    avg_diff = sum(diffs) / len(diffs)
    avg_diff_subindexes = avg_diff
    avg_diff_continuations = None
    print(f"Avg Subindexes Diff: {avg_diff_subindexes}")

Both subindexes and continuations detected
Avg Subindexes Diff: 12.046928937317896, Avg Continuations Diff: 36.01640583918645


In [ ]:
def is_numbers_part(s):
    """
    Returns True if s matches a valid list of numbers and/or ranges separated by commas.
    Examples:
        '150'
        '150, 205, 299'
        '204-208, 324'
        '9, 49, 66–67'
    """
    pattern = re.compile(
        r"""^
        \s*                             # optional leading spaces
        \d+                             # first number
        (?:\s*[–-]\s*\d+)?             # optional range (hyphen or en-dash)
        (?:                            # zero or more additional numbers/ranges
            \s*,\s*                    # comma separator
            \d+                       # next number
            (?:\s*[–-]\s*\d+)?        # optional range
        )*
        \s*$                           # optional trailing spaces
        """,
        re.VERBOSE
    )
    return bool(pattern.match(s))

tests = [
    "150",
    "150, 205, 299",
    "204-208, 324",
    "9, 49, 66–67",
    "abc",
    "150, abc",
    "",
]

for test in tests:
    print(f"{test!r}: {is_numbers_part(test)}")

In [71]:
# index dict will hold the index key and as key a dict, which will have pages or see refs
index_dict = {}
last_main_entry_text = ""
hold_buffer = ""
hold_flag = False
entries_found = 0
for page_num, columns in columnar_data:
    for column in columns:
        for line in column["lines"]:
            x0, y0, x1, y1, line_string = line
            chk, text, pages, see_ref = is_pattern(line_string.strip())
            if chk:
                # determine if main or subindex or continuation
                diff = abs(column["dimensions"][0] - x0)
                if diff <= 2:
                    # main entry point
                    # add to index dict
                    # if previous was buffered, that was wrong entry, ignore
                    if hold_flag:
                        hold_flag = False
                        print(f"Hold Cleared : {hold_buffer}")
                    if see_ref:
                        index_dict[text] = {"pages": None, "see_ref": see_ref}
                    else:
                        index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                        entries_found += len(pages.split(", "))
                    last_main_entry_text = text
                if avg_diff_subindexes is not None and abs(diff - avg_diff_subindexes) <= 2:
                    # subindex
                    if hold_flag:
                        print(f"Hold Addressed subent.: {hold_buffer}")
                        # main entry text didnt have page entries
                        entry = hold_buffer + "!" + text

                    if last_main_entry_text != "":
                        entry = last_main_entry_text + "!" + text
                    # add to last main entry
                    if see_ref:
                        index_dict[entry] = {"pages": None, "see_ref": see_ref}
                    else:
                        index_dict[entry] = {"pages": pages.split(", "), "see_ref": None}
                        entries_found += len(pages.split(", "))
                if avg_diff_continuations is not None and abs(diff - avg_diff_continuations) <= 2:
                    # continuation
                    if hold_flag:
                        
                        entry = hold_buffer + " " + line_string.strip()
                        print(f"Hold Addressed cont.: {entry}")
                        chk, text, pages, see_ref = is_pattern(entry)
                        # add to last main entry
                        if chk:
                            if see_ref:
                                index_dict[text] = {"pages": None, "see_ref": see_ref}
                            else:
                                index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                                entries_found += len(pages.split(", "))

                            last_main_entry_text = text
                        hold_flag = False
                
            else:
                # cases:
                    # only text, because of next line continuation, or possible subentries
                    # only numbers, as a continuation of previous main line entry
                # check second case here, if not, hold in buffer for next line
                chk = is_numbers_part(line_string.strip())
                if chk:
                    print("num check pat")
                    # positive
                    # could be previous hold or main entry
                    if hold_flag:
                        print(f"Hold Addressed in num check: {hold_buffer}")
                        entry = hold_buffer + " " + line_string
                        print(f" --------------------------- {entry}")
                    elif last_main_entry_text!="":
                        entry = last_main_entry_text + " " + line_string
                    chk, text, pages, see_ref = is_pattern(entry.strip())
                    if chk:
                        if see_ref:
                            index_dict[text] = {"pages": None, "see_ref": see_ref}
                        else:
                            index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                            entries_found += len(pages.split(", "))
                        last_main_entry_text = text
                        hold_flag = False
                    else:
                        print(f"Not Pat! Num! Not Pat : {entry}")
                        hold_flag = False
            
                    # did we register some last mainline or subline:
                else:
                    print(f"Hold : {line_string}")

                    # no check, because of maybe next subentry, or continuation
                    hold_flag = True
                    hold_buffer = line_string.strip()



Hold : Index 
Hold : A 
Hold Cleared : A
Hold : Agent, 3, 17, 289, 290, 293, 296, 297, 300, 
Hold Addressed cont.: Agent, 3, 17, 289, 290, 293, 296, 297, 300, 301, 303, 305
Hold : B 
Hold Cleared : B
Hold : Bayesian network, 6, 10, 72, 77, 127, 158, 160, 
num check pat
Hold Addressed in num check: Bayesian network, 6, 10, 72, 77, 127, 158, 160,
 --------------------------- Bayesian network, 6, 10, 72, 77, 127, 158, 160, 198 
Hold : Bellman 
Hold Addressed subent.: Bellman
Hold Addressed subent.: Bellman
Hold Cleared : Bellman
Hold : C 
Hold Cleared : C
Hold : Chain rule for Bayesian networks, 132, 168, 
num check pat
Hold Addressed in num check: Chain rule for Bayesian networks, 132, 168,
 --------------------------- Chain rule for Bayesian networks, 132, 168, 169 
Hold : Convolutional Neural Network (CNN), 277, 
Hold Addressed cont.: Convolutional Neural Network (CNN), 277, 280, 282, 307
Hold : D 
Hold Cleared : D
Hold : E 
Hold Cleared : E
Hold : F 
Hold Cleared : F
Hold : G 
Hold Cl

In [23]:
index_dict

{'A⋆-algorithm': {'pages': ['107', '273'], 'see_ref': None},
 'Action': {'pages': ['97', '98'], 'see_ref': None},
 'Activation function': {'pages': ['248', '258'], 'see_ref': None},
 'Actuators': {'pages': ['17'], 'see_ref': None},
 'Adaptive Resonance Theory (ART)': {'pages': ['286'], 'see_ref': None},
 'Admissible': {'pages': ['107', '109'], 'see_ref': None},
 'Agent, 3, 17, 289, 290, 293, 296, 297, 300, 301, 303, 305': {'pages': ['3',
   '17',
   '289',
   '290',
   '293',
   '296',
   '297',
   '300',
   '301',
   '303',
   '305'],
  'see_ref': None},
 'Agent!autonomous': {'pages': ['10'], 'see_ref': None},
 'Agent!cost-based': {'pages': ['18'], 'see_ref': None},
 'Agent!distributed': {'pages': ['10'], 'see_ref': None},
 'Agent!goal-based': {'pages': ['17'], 'see_ref': None},
 'Agent!hardware': {'pages': ['17'], 'see_ref': None},
 'Agent!intelligent': {'pages': ['17'], 'see_ref': None},
 'Agent!learning': {'pages': ['11', '18', '178'], 'see_ref': None},
 'Agent!reﬂex': {'pages': ['

In [21]:
entries_found

726

In [ ]:
# Pattern Matchers

def is_pattern(text):
    pattern = re.compile(
    r"""
    ^(.+?),                # group 1: text (greedy, up to last comma before numbers)
    \s*                    # optional spaces
    ((?:\d+(?:–\d+)?       # one number or range (e.g., 150 or 204–208)
        (?:,\s*\d+(?:–\d+)?)*))  # optionally more numbers/ranges separated by commas
    $                      # end of string
    """,
    re.VERBOSE
    )

    pattern = re.compile(
        r"""
        ^\s*                                # optional leading spaces
        (?P<entry>.+?)                      # capture entry text (lazy)
        # \s*,?\s*                            # optional comma before numbers
        \s*,?\s+                     # **at least one space** before numbers, optional comma allowed here
        ((?P<pages>(\d+(?:\s*[–-]\s*\d+)?   # a number or range
                (?:\s*,\s*\d+(?:\s*[–-]\s*\d+)?)*   # more numbers/ranges separated by commas
                )
        )
        |
        (?:see\s+                  # OR the cross-reference "see" keyword (case-insensitive)
            (?P<see_ref>.+)         # cross-reference text (rest of line)
        ))
        
        \s*$                               # optional trailing spaces
        """,
        re.IGNORECASE | re.VERBOSE
    )
    if '©' in text:
        return False, None, None, None
    match = pattern.match(text)
    if match:
        text= match.group("entry")
        pages = match.group("pages")
        see_ref = match.group("see_ref")
        return True, text, pages, see_ref
    else:
        return False, None, None, None
    

def is_numbers_part(s):
    """
    Returns True if s matches a valid list of numbers and/or ranges separated by commas.
    Examples:
        '150'
        '150, 205, 299'
        '204-208, 324'
        '9, 49, 66–67'
    """
    pattern = re.compile(
        r"""^
        \s*                             # optional leading spaces
        \d+                             # first number
        (?:\s*[–-]\s*\d+)?             # optional range (hyphen or en-dash)
        (?:                            # zero or more additional numbers/ranges
            \s*,\s*                    # comma separator
            \d+                       # next number
            (?:\s*[–-]\s*\d+)?        # optional range
        )*
        \s*$                           # optional trailing spaces
        """,
        re.VERBOSE
    )
    return bool(pattern.match(s))




In [72]:
def get_line_indexes(PDF_PATH:str):
    doc = fitz.open(PDF_PATH)
    final_list = []
    x0 = -1
    y0 = -1
    x1 = -1
    y1 = -1 # working coordinates
    current_line = -1
    line_string = ""
    current_block = -1
    for page_num, page in enumerate(doc, start=1):
        print(f"--- Page {page_num} ---")
        text = page.get_text()
        # print(text)
        text = page.get_text("words")  # extracts text in reading order
        current_line = -1
        line_string = ""
        current_block = -1
        
        page_data = []
        for word in text:
            x0_, y0_, x1_, y1_, word_str, block_no, line_no, word_no = word
            print(word)
            # sometimes pymupdf gives different lin index to things that are on same line, manuakl check with x and y coords
            flag = ((abs(y0_ - y0) < 5) and x0_ -x1 <= 50) or line_no == current_line   # flag true means same line

            if block_no != current_block or (not flag ):
                # save current line before moving to next
                if current_line != -1:
                    page_data.append((x0, y0, x1, y1, line_string))
                
                current_block = block_no
                current_line = line_no
                line_string = ""
                x0, y0, x1, y1 = x0_, y0_, x1_, y1_
            
            # appending word and managing coordinates
            line_string += word_str + " "
            x1 = max(x1, x1_)
            y1 = max(y1, y1_)
            x0 = min(x0, x0_)
            y0 = min(y0, y0_)

        page_data.append((x0, y0, x1, y1, line_string))
        final_list.append((page_num, page_data))

    return final_list
    


def make_columns(final_list):
    columnar_data = [] # (page number,  columns = {column_dims = [], column_lines = []} )

    current_column = [-1,-1,-1,-1]
    page_columns = []
    column_data =[]
    is_valid = False
    is_footer = False
    for page_num, page_data in final_list:
        is_valid = False
        print(f"Processing page {page_num}")
        for line in page_data:
            x0, y0, x1, y1, line_string = line

            # print(f"Line: {line_string.strip()} at ({x0}, {y0}, {x1}, {y1})")
            # # check if line fits in current column
            if current_column[0] == -1:
                # initialize column
                print("New column detected")
                print(f"Current column: {current_column}")
                current_column = [x0,y0,x1,y1]
                print(f"Starting new column with line at ({x0}, {y0}, {x1}, {y1}) with text: {line_string.strip()}")
                if '©' in line_string:
                    is_footer = True
                is_valid = False
                column_data = []
                column_data.append(line)
                chk, _, _,_ = is_pattern(line_string.strip())
                is_valid = is_valid or chk

            else:
                # check if line fits in current column

                if ((current_column[0] <= x0 and x0  <= current_column[2] ) or (current_column[0] <= x1 and x1 <= current_column[2]) or (x0 <= current_column[0] and current_column[2] <= x1)) and \
                    (current_column[0]- x0 <= 50):
                    # print("Fits in current column")
                    # update column coordinates
                    current_column[0] = min(current_column[0], x0)
                    current_column[2] = max(current_column[2], x1)
                    current_column[1] = min(current_column[1], y0)
                    current_column[3] = max(current_column[3], y1)
                    column_data.append(line)
                    chk, _, _,_ = is_pattern(line_string.strip())
                    if '©' in line_string:
                        is_footer = True
                    is_valid = is_valid or chk
                else:
                    # save current column and start new one
                    print("New column detected")
                    print(f"Current column: {current_column}")
                    print(f"Starting new column with line at ({x0}, {y0}, {x1}, {y1}) with text: {line_string.strip()}")
                    if is_valid and not is_footer:
                        page_columns.append({"dimensions" : current_column, "lines": column_data})
                    current_column = [x0,y0,x1,y1]
                    is_valid = False
                    is_footer = False
                    column_data = []
                    column_data.append(line)
                    chk, _, _, _ = is_pattern(line_string.strip())
                    is_valid = is_valid or chk
                    if '©' in line_string:
                        is_footer = True
        # save last column of the page
        if is_valid and not is_footer:
            page_columns.append({"dimensions" : current_column, "lines": column_data})
        is_valid = False
        is_footer = False
        print(f"End of page {page_num}, columns: {page_columns}")
        if current_column[0] != -1:
            columnar_data.append([page_num, page_columns])
            current_column = [-1,-1,-1,-1]
        page_columns = []

    return columnar_data


def get_indentations(columnar_data):
    diffs = []
    for page_data in columnar_data:
        print(f"Page {page_data[0]} has {len(page_data[1])} columns.")
        for column in page_data[1]:
            print("Column Coordinates:", column["dimensions"])
            x_main = column["dimensions"][0]
            for line in column["lines"]:
                print(" Line:", line)
                x_line = line[0]
                diffs.append(abs(x_main - x_line))

    # remove diffs less than 1
    diffs = [d for d in diffs if d >= 1]
    diffs.sort()

    # check first and last diff to see range
    min_diff = diffs[0]
    max_diff = diffs[-1]
    # subtract min_diff from max_diff
    range_diff = max_diff - min_diff

    # if range is greater than 2, then we have subindexes and continuations, else only subindexes
    if range_diff > 2:
        print("Both subindexes and continuations detected")
        # separate diffs into two groups based on a threshold
        subindexes = [d for d in diffs if abs(d - min_diff) <= 2]
        continuations = [d for d in diffs if abs(d - max_diff) <= 2]
        # average diff for subindexes
        avg_diff_subindexes = sum(subindexes) / len(subindexes)
        avg_diff_subindexes
        # average diff for continuations
        avg_diff_continuations = sum(continuations) / len(continuations)
        avg_diff_continuations
        print(f"Avg Subindexes Diff: {avg_diff_subindexes}, Avg Continuations Diff: {avg_diff_continuations}")
    else:
        print("Only subindexes detected")
        # average diff
        avg_diff = sum(diffs) / len(diffs)
        avg_diff_subindexes = avg_diff
        avg_diff_continuations = None
        print(f"Avg Subindexes Diff: {avg_diff_subindexes}")

    return avg_diff_subindexes, avg_diff_continuations




def get_index_list(columnar_data):
    # index dict will hold the index key and as key a dict, which will have pages or see refs
    index_dict = {}
    last_main_entry_text = ""
    hold_buffer = ""
    hold_flag = False
    entries_found = 0

    # get subindex and continuation differences
    avg_diff_subindexes, avg_diff_continuations = get_indentations(columnar_data)

    for page_num, columns in columnar_data:
        for column in columns:
            for line in column["lines"]:
                x0, y0, x1, y1, line_string = line
                chk, text, pages, see_ref = is_pattern(line_string.strip())
                if chk:
                    # determine if main or subindex or continuation
                    diff = abs(column["dimensions"][0] - x0)
                    if diff <= 2:
                        # main entry point
                        # add to index dict
                        # if previous was buffered, that was wrong entry, ignore
                        if hold_flag:
                            hold_flag = False
                            print(f"Hold Cleared : {hold_buffer}")
                        if see_ref:
                            index_dict[text] = {"pages": None, "see_ref": see_ref}
                        else:
                            index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                            entries_found += len(pages.split(", "))
                        last_main_entry_text = text
                    if avg_diff_subindexes is not None and abs(diff - avg_diff_subindexes) <= 2:
                        # subindex
                        if hold_flag:
                            print(f"Hold Addressed subent.: {hold_buffer}")
                            # main entry text didnt have page entries
                            entry = hold_buffer + "!" + text

                        if last_main_entry_text != "":
                            entry = last_main_entry_text + "!" + text
                        # add to last main entry
                        if see_ref:
                            index_dict[entry] = {"pages": None, "see_ref": see_ref}
                        else:
                            index_dict[entry] = {"pages": pages.split(", "), "see_ref": None}
                            entries_found += len(pages.split(", "))
                    if avg_diff_continuations is not None and abs(diff - avg_diff_continuations) <= 2:
                        # continuation
                        if hold_flag:
                            
                            entry = hold_buffer + " " + line_string.strip()
                            print(f"Hold Addressed cont.: {entry}")
                            chk, text, pages, see_ref = is_pattern(entry)
                            # add to last main entry
                            if chk:
                                if see_ref:
                                    index_dict[text] = {"pages": None, "see_ref": see_ref}
                                else:
                                    index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                                    entries_found += len(pages.split(", "))

                                last_main_entry_text = text
                            hold_flag = False
                    
                else:
                    # cases:
                        # only text, because of next line continuation, or possible subentries
                        # only numbers, as a continuation of previous main line entry
                    # check second case here, if not, hold in buffer for next line
                    chk = is_numbers_part(line_string.strip())
                    if chk:
                        print("num check pat")
                        # positive
                        # could be previous hold or main entry
                        if hold_flag:
                            print(f"Hold Addressed in num check: {hold_buffer}")
                            entry = hold_buffer + " " + line_string
                            print(f" --------------------------- {entry}")
                        elif last_main_entry_text!="":
                            entry = last_main_entry_text + " " + line_string
                        chk, text, pages, see_ref = is_pattern(entry.strip())
                        if chk:
                            if see_ref:
                                index_dict[text] = {"pages": None, "see_ref": see_ref}
                            else:
                                index_dict[text] = {"pages": pages.split(", "), "see_ref": None}
                                entries_found += len(pages.split(", "))
                            last_main_entry_text = text
                            hold_flag = False
                        else:
                            print(f"Not Pat! Num! Not Pat : {entry}")
                            hold_flag = False
                
                        # did we register some last mainline or subline:
                    else:
                        print(f"Hold : {line_string}")

                        # no check, because of maybe next subentry, or continuation
                        hold_flag = True
                        hold_buffer = line_string.strip()

    return index_dict


In [9]:


import fitz
import re

def extract_index(BOOK_PDF:str):
    index_lines = get_line_indexes(BOOK_PDF)
    index_columns = make_columns(index_lines)
    index_dict = get_index_list(index_columns)

    return index_dict
    


In [32]:
def get_total_entry_count(index_dict):
    total_count = 0
    for key in index_dict:
        value = index_dict[key]
        pages = value['pages']

        total_count += len(pages) if pages is not None else 0

    print("Total Index Entries : ", total_count)

In [73]:
ai_index_list = extract_index(ai_pdf_path)
get_total_entry_count(ai_index_list)


--- Page 1 ---
(53.85820007324219, 67.56885528564453, 93.5017318725586, 83.5091552734375, 'Index', 0, 0, 0)
(53.85820007324219, 190.1238555908203, 59.93836975097656, 198.59205627441406, 'A', 1, 0, 0)
(53.85820007324219, 198.47933959960938, 100.3957290649414, 209.71176147460938, 'A⋆-algorithm,', 1, 1, 0)
(103.29185485839844, 201.2401123046875, 118.1112060546875, 209.70831298828125, '107,', 1, 1, 1)
(120.92264556884766, 201.2401123046875, 133.62493896484375, 209.70831298828125, '273', 1, 1, 2)
(53.85449981689453, 211.16485595703125, 78.79335021972656, 219.633056640625, 'Action,', 1, 2, 0)
(81.63019561767578, 211.16485595703125, 92.24932098388672, 219.633056640625, '97,', 1, 2, 1)
(95.06076049804688, 211.16485595703125, 103.52896118164062, 219.633056640625, '98', 1, 2, 2)
(53.846031188964844, 221.089599609375, 89.17536163330078, 229.55780029296875, 'Activation', 1, 3, 0)
(92.05455017089844, 221.089599609375, 122.10819244384766, 229.55780029296875, 'function,', 1, 3, 1)
(124.93656921386719

In [74]:
ai_index_list

{'A⋆-algorithm': {'pages': ['107', '273'], 'see_ref': None},
 'Action': {'pages': ['97', '98'], 'see_ref': None},
 'Activation function': {'pages': ['248', '258'], 'see_ref': None},
 'Actuators': {'pages': ['17'], 'see_ref': None},
 'Adaptive Resonance Theory (ART)': {'pages': ['286'], 'see_ref': None},
 'Admissible': {'pages': ['107', '109'], 'see_ref': None},
 'Agent': {'pages': ['3',
   '17',
   '289',
   '290',
   '293',
   '296',
   '297',
   '300',
   '301',
   '303',
   '305'],
  'see_ref': None},
 'Agent!autonomous': {'pages': ['10'], 'see_ref': None},
 'Agent!cost-based': {'pages': ['18'], 'see_ref': None},
 'Agent!distributed': {'pages': ['10'], 'see_ref': None},
 'Agent!goal-based': {'pages': ['17'], 'see_ref': None},
 'Agent!hardware': {'pages': ['17'], 'see_ref': None},
 'Agent!intelligent': {'pages': ['17'], 'see_ref': None},
 'Agent!learning': {'pages': ['11', '18', '178'], 'see_ref': None},
 'Agent!reﬂex': {'pages': ['17'], 'see_ref': None},
 'Agent!software': {'pages':

In [40]:
index_list = extract_index(assmb_pdf_path)
get_total_entry_count(index_list)

--- Page 1 ---
(53.85820007324219, 71.19389343261719, 93.5017318725586, 87.13419342041016, 'Index', 0, 0, 0)
(53.85820007324219, 191.26426696777344, 59.93836975097656, 199.7324676513672, 'A', 1, 0, 0)
(53.85820007324219, 201.23995971679688, 84.17435455322266, 209.70816040039062, 'Absolute', 1, 1, 0)
(87.0239028930664, 201.23995971679688, 114.25762176513672, 209.70816040039062, 'address,', 1, 1, 1)
(117.12750244140625, 201.23995971679688, 129.82980346679688, 209.70816040039062, '274', 1, 1, 2)
(53.85819625854492, 210.80239868164062, 71.055419921875, 219.6295166015625, 'add,', 1, 2, 0)
(73.87024688720703, 211.16131591796875, 84.4910659790039, 219.6295166015625, '29,', 1, 2, 1)
(87.30589294433594, 211.16131591796875, 97.8699722290039, 219.6295166015625, '30,', 1, 2, 2)
(100.68479919433594, 211.16131591796875, 115.50076293945312, 219.6295166015625, '282,', 1, 2, 3)
(118.37232971191406, 211.16131591796875, 131.0746307373047, 219.6295166015625, '334', 1, 2, 4)
(53.85565948486328, 221.0826721

In [36]:
index_list = extract_index(alg_pdf_path)
get_total_entry_count(index_list)

--- Page 1 ---
(53.15250015258789, 102.49510192871094, 121.93174743652344, 127.28219604492188, 'Index', 0, 0, 0)
(53.15250015258789, 178.03179931640625, 89.60565948486328, 188.51258850097656, 'ϵ-moves,', 1, 0, 0)
(92.92320251464844, 178.54998779296875, 107.86710357666016, 188.51258850097656, '703', 1, 0, 1)
(53.15250015258789, 190.54495239257812, 130.8109588623047, 200.50755310058594, '#P-completeness,', 1, 1, 0)
(134.1384735107422, 190.54495239257812, 151.8419952392578, 200.50755310058594, '476,', 1, 1, 1)
(155.1595458984375, 190.54495239257812, 170.10342407226562, 200.50755310058594, '548', 1, 1, 2)
(53.15250015258789, 202.5399169921875, 68.09639739990234, 212.5025177001953, '0/1', 1, 2, 0)
(71.4139404296875, 202.5399169921875, 111.01527404785156, 212.5025177001953, 'knapsack', 1, 2, 1)
(114.33281707763672, 202.5399169921875, 152.50950622558594, 212.5025177001953, 'problem,', 1, 2, 2)
(155.82705688476562, 202.5399169921875, 170.77093505859375, 212.5025177001953, '497', 1, 2, 3)
(53.1

In [37]:
index_list = extract_index(cyber_pdf_path)
get_total_entry_count(ai_index_list)

--- Page 1 ---
(54.119300842285156, 48.245079040527344, 92.7742919921875, 71.5018310546875, 'Index', 0, 0, 0)
(54.119300842285156, 224.9517364501953, 84.70642852783203, 237.3068389892578, 'Symbols', 1, 0, 0)
(54.119300842285156, 248.64688110351562, 73.1473617553711, 257.1150817871094, '0-day', 2, 0, 0)
(75.26441192626953, 248.64688110351562, 95.31713104248047, 257.1150817871094, 'attack', 2, 0, 1)
(105.2757339477539, 248.64688110351562, 120.09508514404297, 257.1150817871094, '295,', 2, 1, 0)
(121.7887191772461, 248.64688110351562, 134.49102783203125, 257.1150817871094, '323', 2, 1, 1)
(54.119300842285156, 258.65087890625, 74.81560516357422, 267.11907958984375, '3-way', 2, 2, 0)
(76.93265533447266, 258.65087890625, 112.50758361816406, 267.11907958984375, 'handshake', 2, 2, 1)
(122.46619415283203, 258.65087890625, 135.16848754882812, 267.11907958984375, '138', 2, 3, 0)
(54.119300842285156, 268.6548767089844, 74.34983825683594, 277.1230773925781, '3DES', 2, 4, 0)
(84.30843353271484, 268.6

In [38]:
index_list = extract_index(ds_pdf_path)
get_total_entry_count(index_list)

--- Page 1 ---
(79.93990325927734, 86.73210144042969, 148.72410583496094, 111.51920318603516, 'Index', 0, 0, 0)
(79.93990325927734, 162.0789031982422, 99.4466781616211, 172.04150390625, 'A/B', 1, 0, 0)
(102.76422882080078, 162.0789031982422, 134.87368774414062, 172.04150390625, 'testing,', 1, 0, 1)
(138.20120239257812, 162.0789031982422, 148.16378784179688, 172.04150390625, '86', 1, 0, 2)
(79.93990325927734, 174.03392028808594, 106.799072265625, 183.99652099609375, 'Aaron', 1, 1, 0)
(110.11661529541016, 174.03392028808594, 149.13015747070312, 183.99652099609375, 'Schwartz', 1, 1, 1)
(152.44769287109375, 174.03392028808594, 172.960693359375, 183.99652099609375, 'case,', 1, 1, 2)
(176.2882080078125, 174.03392028808594, 186.25079345703125, 183.99652099609375, '68', 1, 1, 3)
(79.93990325927734, 185.9889373779297, 94.46537780761719, 195.9515380859375, 'AB', 1, 2, 0)
(97.78292083740234, 185.9889373779297, 129.8923797607422, 195.9515380859375, 'testing,', 1, 2, 1)
(133.21990966796875, 185.988

In [43]:
def handle_ranges(page_list):
    """
    Given a list of page numbers and ranges as strings, for the ranges, only keep the first term
    Example:
        Input: ['150', '204-208', '299']
        Output: [150, 204, 299]
    """
    result = []
    for item in page_list:
        if '–' in item:
            start, end = item.split('–')
            result.append(int(start.strip()))
        elif '-' in item:
            start, end = item.split('-')
            result.append(int(start.strip()))
        else:
            result.append(int(item.strip()))
    return result

In [44]:
def index_cleaner(index_dict):
    cleaned_index = {}
    for key in index_dict:
        value = index_dict[key]
        pages = value['pages']
        see_ref = value['see_ref']
        if pages is not None:
            cleaned_pages = handle_ranges(pages)
            cleaned_index[key] = {'pages': cleaned_pages, 'see_ref': see_ref}
        else:
            cleaned_index[key] = {'pages': None, 'see_ref': see_ref}
    return cleaned_index

In [45]:
index_cleaner(index_list)

{'Absolute address': {'pages': [274], 'see_ref': None},
 'add': {'pages': [29, 30, 282, 334], 'see_ref': None},
 'Addition': {'pages': [314], 'see_ref': None},
 'Addition instructions': {'pages': [31], 'see_ref': None},
 'ADDR operator': {'pages': [18], 'see_ref': None},
 'Aliasing': {'pages': [160], 'see_ref': None},
 'American Standard Code for Information Interchange (ASCII), 311, 312': {'pages': [311,
   312],
  'see_ref': None},
 'and': {'pages': [323], 'see_ref': None},
 'And operator (&&)': {'pages': [62], 'see_ref': None},
 'Arithmetic instructions': {'pages': [29], 'see_ref': None},
 'Arithmetic shift': {'pages': [105], 'see_ref': None},
 '.asm': {'pages': [291], 'see_ref': None},
 'Array of strings': {'pages': [204], 'see_ref': None},
 'Arrays': {'pages': [159], 'see_ref': None},
 'Arrays!64-bit arrays': {'pages': [258], 'see_ref': None},
 'Arrays!ﬂoating-point arrays': {'pages': [232], 'see_ref': None},
 'Assembler': {'pages': [1], 'see_ref': None},
 'Assembly language': {'p

In [66]:
from tqdm import tqdm
# try to cite these in books now.
def find_closest_page(page, page_breaks, page_positions, book_len, is_forward=True):
    print(f"Finding closest page for page {page}, is_forward={is_forward}")
    print(f"Page breaks available: {page_breaks[:5] if len(page_breaks) > 5 else page_breaks}... (total: {len(page_breaks)})")
    
    if str(page) in page_breaks:
        print(f"Exact match found for page {page}")
        return page_positions[page]
    
    if is_forward:
        bound = book_len
        forward = page
        while forward <= bound:
            if str(forward) in page_breaks:
                print(f"Forward match found: {forward}")
                return page_positions[forward]
            forward += 1
        print(f"No forward match found for page {page}")
        return -1
    else:
        bound = 0
        backward = page
        while backward >= bound:
            if str(backward) in page_breaks:
                print(f"Backward match found: {backward}")
                return page_positions[backward]
            backward -= 1
        print(f"No backward match found for page {page}")
        return 0  # No valid page found


def add_indexes(latex_content, index, book_len):
    print(f"Adding indexes to LaTeX content. Index has {len(index)} terms.")
    matched = 0
    not_matched = 0
    not_found_terms = {}  # Dictionary to store terms not found along with page numbers

    # Extract page breaks once to avoid repeated searches
    print("Extracting page breaks from LaTeX content...")
    page_breaks = re.findall(r'%---- Page End Break Here ---- Page : (\d+)', latex_content)
    page_positions = {int(page): pos for page, pos in zip(page_breaks, [m.start() for m in re.finditer(r'%---- Page End Break Here ---- Page : \d+', latex_content)])}
    print(f"Found {len(page_breaks)} page breaks in the LaTeX content")
    
    if not page_breaks:
        print("WARNING: No page breaks found in LaTeX content. Check page break format.")
    
    # Debug information for page positions
    if page_positions:
        print(f"Page position examples: {list(page_positions.items())[:3]}")
    
    for index_term, values in tqdm(index.items()):
        if values['pages'] is None:
            continue  # Skip see references for now
        pages = values['pages']
        for page in pages:
            # Debug for specific terms, if needed
            # debug = (index_term == 'application')  # Example term to debug
            debug = False
            
            if debug:
                print(f"DEBUG: Processing '{index_term}' on page {page}")
            
            try:
                # Find page boundaries
                upper_bound = find_closest_page(page+1, page_breaks, page_positions, book_len, True)
                lower_bound = find_closest_page(page-2, page_breaks, page_positions, False)
                
                if debug:
                    print(f"DEBUG: Page {page} bounds - lower: {lower_bound}, upper: {upper_bound}")
                
                if upper_bound == -1 or lower_bound == 0:
                    print(f"Warning: Could not find proper bounds for page {page} with term '{index_term}'")
                    if index_term not in not_found_terms:
                        not_found_terms[index_term] = []
                    not_found_terms[index_term].append(page)
                    not_matched += 1
                    continue
                
                page_content = latex_content[lower_bound:upper_bound]
                
                # Extract the search term (for subindex entries)
                term = index_term
                if '!' in index_term:
                    # subindex entry
                    term = index_term.split("!", 1)[1]
                
                if debug:
                    print(f"DEBUG: Searching for term '{term}' (from '{index_term}')")
                
                match = re.search(re.escape(term), page_content, re.IGNORECASE)

                if match:
                    if debug:
                        print(f"DEBUG: Found match at position {match.start()} in page content")
                    
                    # Look for the term inside braces or brackets
                    pattern = (
                        r"\{[^{}]*" + re.escape(term) + r"[^{}]*\}" +
                        r"|" +  # OR
                        r"\[[^\[\]]*" + re.escape(term) + r"[^\[\]]*\]"
                    )
                    brace_match = re.search(pattern, page_content, re.IGNORECASE)
                    
                    if brace_match:
                        # Term is inside a command
                        if debug:
                            print(f"DEBUG: Term found inside braces/brackets at position {brace_match.start()}")
                        
                        term_end = lower_bound + brace_match.end()
                        indexed_term = "\\index{" + index_term + "}"
                        
                        # Add debug check to see what we're inserting and where
                        if debug:
                            context_before = latex_content[term_end-10:term_end]
                            context_after = latex_content[term_end:term_end+10]
                            print(f"DEBUG: Inserting '{indexed_term}' at position {term_end}")
                            print(f"DEBUG: Context: ...{context_before}|HERE|{context_after}...")
                        
                        newline_pos = latex_content.find("\n", term_end)
                        if newline_pos != -1 and term_end < newline_pos:
                            term_end = newline_pos
                        
                        latex_content = latex_content[:term_end] + indexed_term + latex_content[term_end:]

                    
                    else:
                        # Term is not inside a command
                        if debug:
                            print(f"DEBUG: Term found in regular text")
                        
                        term_end = lower_bound + match.end()
                        indexed_term = "\\index{" + index_term + "}"
                        
                        # Add debug check to see what we're inserting and where
                        if debug:
                            context_before = latex_content[term_end-10:term_end]
                            context_after = latex_content[term_end:term_end+10]
                            print(f"DEBUG: Inserting '{indexed_term}' at position {term_end}")
                            print(f"DEBUG: Context: ...{context_before}|HERE|{context_after}...")
                        
                        newline_pos = latex_content.find("\n", term_end)
                        if newline_pos != -1 and term_end < newline_pos:
                            term_end = newline_pos

                        latex_content = latex_content[:term_end] + indexed_term + latex_content[term_end:]
                    
                    matched += 1
                    
                    # Progress update
                    if matched % 100 == 0:
                        print(f"Matched {matched} terms so far")
                        
                else:
                    if debug:
                        print(f"DEBUG: No match found for term '{term}' on page {page}")
                    
                    # Record not found term
                    if index_term not in not_found_terms:
                        not_found_terms[index_term] = []
                    not_found_terms[index_term].append(page)
                    not_matched += 1
                    
                    # Progress update
                    if not_matched % 100 == 0:
                        print(f"Not matched {not_matched} terms so far")
                        
            except Exception as e:
                print(f"Error processing term '{index_term}' on page {page}: {e}")
                if index_term not in not_found_terms:
                    not_found_terms[index_term] = []
                not_found_terms[index_term].append(page)
                not_matched += 1
    
    print(f"Matched: {matched}, Not Matched: {not_matched}")

    # handle not found terms
    print(f"Processing {len(not_found_terms)} terms that were not found")
    page_based_terms = {}

    # Iterate through the original not_found_terms dictionary
    for index_term, pages in not_found_terms.items():
        for page in pages:
            if page not in page_based_terms:
                page_based_terms[page] = []  # Initialize list if page is not already in the dictionary
            page_based_terms[page].append(index_term)  # Add the index term to the list for the current page

    print(f"Grouping not found terms by {len(page_based_terms)} pages")
    for page, not_found_index_terms in page_based_terms.items():
        print(f"Adding {len(not_found_index_terms)} not found terms to page {page}")
        
        index_string = ""
        index_string = "".join([f"\\index{{{term}}}" for term in not_found_index_terms])

        # Re-find the page breaks and positions
        try:
            page_breaks = re.findall(r'%---- Page End Break Here ---- Page : (\d+)', latex_content)
            page_positions = {int(page): pos for page, pos in zip(page_breaks, [m.start() for m in re.finditer(r'%---- Page End Break Here ---- Page : \d+', latex_content)])}

            upper_bound = find_closest_page(page+0, page_breaks, page_positions, book_len, True)
            lower_bound = find_closest_page(page-1, page_breaks, page_positions, False)

            if upper_bound == -1 or lower_bound == 0:
                print(f"Warning: Could not find proper bounds for page {page} when adding not found terms")
                continue
                
            page_content = latex_content[lower_bound:upper_bound]
            index_position = lower_bound + (upper_bound - lower_bound) // 2

            # if index_position is before \begin{document}, move it to after
            begin_doc_pos = latex_content.find(r"\begin{document}")
            if begin_doc_pos != -1 and index_position < begin_doc_pos:
                index_position = begin_doc_pos + len(r"\begin{document}")

            # Move forward until a newline
            next_newline_pos = latex_content.find("\n", index_position)
            if next_newline_pos == -1:
                print(f"Warning: No newline found after position {index_position}")
                next_newline_pos = upper_bound  # In case no newline is found, go till the end of the content

            # Debug info to see what we're inserting and where
            try:
                context_before = latex_content[next_newline_pos-10:next_newline_pos]
                context_after = latex_content[next_newline_pos:next_newline_pos+10]
                print(f"Inserting {len(not_found_index_terms)} index entries at position {next_newline_pos}")
                print(f"Context: ...{context_before}|HERE|{context_after}...")
            except Exception as e:
                print(f"Error showing context: {e}")
            
            latex_content = latex_content[:next_newline_pos] + index_string + latex_content[next_newline_pos:]
        
        except Exception as e:
            print(f"Error processing not found terms for page {page}: {e}")
    print(f"Matched: {matched}, Not Matched: {not_matched}")

    # Clean up the LaTeX content
    return latex_content, not_found_terms


In [67]:
def driver(pdf_path , latex_path, index_dict, output_path):
    with open(latex_path, 'r', encoding='utf-8') as f:
        latex_content = f.read()
    
    #open pdf
    doc = fitz.open(pdf_path)
    book_len = doc.page_count
    print(f"Book LaTeX content length: {book_len} characters")
    
    updated_latex, not_found_terms = add_indexes(latex_content, index_dict, book_len)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(updated_latex)
    
    print(f"Updated LaTeX content written to {output_path}")
    print(f"Total terms not found: {len(not_found_terms)}")
    return updated_latex, not_found_terms

In [81]:
pdf_path = ai_pdf_path
tex_path = ai_tex_path
index_list = extract_index(pdf_path)
book_path = ai_book_path


--- Page 1 ---
(53.85820007324219, 67.56885528564453, 93.5017318725586, 83.5091552734375, 'Index', 0, 0, 0)
(53.85820007324219, 190.1238555908203, 59.93836975097656, 198.59205627441406, 'A', 1, 0, 0)
(53.85820007324219, 198.47933959960938, 100.3957290649414, 209.71176147460938, 'A⋆-algorithm,', 1, 1, 0)
(103.29185485839844, 201.2401123046875, 118.1112060546875, 209.70831298828125, '107,', 1, 1, 1)
(120.92264556884766, 201.2401123046875, 133.62493896484375, 209.70831298828125, '273', 1, 1, 2)
(53.85449981689453, 211.16485595703125, 78.79335021972656, 219.633056640625, 'Action,', 1, 2, 0)
(81.63019561767578, 211.16485595703125, 92.24932098388672, 219.633056640625, '97,', 1, 2, 1)
(95.06076049804688, 211.16485595703125, 103.52896118164062, 219.633056640625, '98', 1, 2, 2)
(53.846031188964844, 221.089599609375, 89.17536163330078, 229.55780029296875, 'Activation', 1, 3, 0)
(92.05455017089844, 221.089599609375, 122.10819244384766, 229.55780029296875, 'function,', 1, 3, 1)
(124.93656921386719

In [85]:
doc = fitz.open(book_path)
doc.page_count

357

In [ ]:
new_latex, not_found = driver(book_path, tex_path, index_cleaner(index_list), "AI_added_latex.tex")



Book LaTeX content length: 357 characters
Adding indexes to LaTeX content. Index has 498 terms.
Extracting page breaks from LaTeX content...
Found 232 page breaks in the LaTeX content
Page position examples: [(2, 3056), (6, 9906), (10, 18421)]


100%|██████████| 498/498 [00:00<00:00, 6677.21it/s]

Finding closest page for page 108, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 108
Finding closest page for page 105, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 105
Finding closest page for page 274, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 274
Finding closest page for page 271, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 271
Finding closest page for page 98, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 98
Finding closest page for page 95, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 95
Finding closest page for page 99, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']...

Adding 4 not found terms to page 8
Finding closest page for page 8, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Forward match found: 10
Finding closest page for page 7, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
No forward match found for page 7
Inserting 4 index entries at position 9901
Context: ...ex{PROLOG}|HERE|
10.3 Unin...
Adding 5 not found terms to page 25
Finding closest page for page 25, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 25
Finding closest page for page 24, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match found for page 24
Inserting 5 index entries at position 60458
Context: ...sed agent.|HERE|

Analogou...
Adding 1 not found terms to page 112
Finding closest page for page 112, is_forward=True
Page breaks available: ['2', '6', '10', '11', '12']... (total: 232)
Exact match fo

In [ ]:
# find count of \index in new_latex
index_count = len(re.findall(r'\\index\{', new_latex))
print(f"Total \\index entries in updated LaTeX: {index_count}")

Total \index entries in updated LaTeX: 726


In [ ]:
doc = fitz.open(pdf_path)


In [ ]:
for page_num, columns in page_columns.items():
    page = doc[page_num - 1]  # 0-based index in PyMuPDF
    for coords in columns:
        rect = fitz.Rect(*coords)
        # Draw rectangle (stroke only, no fill)
        page.draw_rect(rect, color=(1, 0, 0), width=1.5)  # red outline

# Save to a new file
doc.save(f"{bk}_with_columns_highlighted.pdf")
doc.close()

In [ ]:
import re

pattern = re.compile(
    r"""
    ^(.+?),                # group 1: text (greedy, up to last comma before numbers)
    \s*                    # optional spaces
    ((?:\d+(?:–\d+)?       # one number or range (e.g., 150 or 204–208)
        (?:,\s*\d+(?:–\d+)?)*))  # optionally more numbers/ranges separated by commas
    $                      # end of string
    """,
    re.VERBOSE
)


examples = [
    "Bayes’ theorem, 150, 205, 299",
    "Babbage, Charles, 57, 90",
    "cmpsb, 198, 204–208, 324",
    "eflags register, 9, 49, 66–67",
    
    "350"
]

for ex in examples:
    match = pattern.match(ex)
    if match:
        text, pages = match.groups()
        print(f"Text: {text}\nPages: {pages.split(', ')}\n")
    else:
        print(f"No match for: {ex}\n")